# MSR-VTT training

Public adaptation of the historical Kaggle workflow. Python modules are byte-identical to captured sources; this notebook replaces embedded source/data payloads with verified files. It has no execution outputs and has not been rerun as a GPU job. Read docs/protocols/kaggle_research_sources.md before running. Attach authorized inputs separately; no test-based model selection.

In [ ]:
# Cell 1 — Không cài đè torch/CUDA/numpy/OpenCV của Kaggle.
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

base = Path(os.environ.get("MSRVTT_WORKSPACE", "/kaggle/working/msrvtt_b3_fulltrain"))
CODE_DIR = base / "code"
CODE_DIR.mkdir(parents=True, exist_ok=True)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
for name in ("torch", "numpy", "cv2", "PIL"):
    if importlib.util.find_spec(name) is None:
        raise RuntimeError(f"Thiếu {name}: chọn image Python/GPU chuẩn của Kaggle.")
packages = {"ftfy": "ftfy==6.3.1", "regex": "regex==2026.9.29", "tqdm": "tqdm==4.70.1"}
missing = [p for m, p in packages.items() if importlib.util.find_spec(m) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                           "--disable-pip-version-check", *missing])
torch = importlib.import_module("torch")

print("Python", sys.version.split()[0], "torch", torch.__version__,
      "CUDA", torch.cuda.is_available())


In [ ]:
# Cấu hình lấy từ lượt chạy đã lưu; các đường dẫn dữ liệu được để trống.
CONFIG = {'INPUT_ROOT': '/kaggle/input',
 'WORK_DIR': '',
 'VIDEO_ROOT': '',
 'TEST_CSV': '',
 'TRAIN_CSV': '',
 'CAPTIONS_JSON': '',
 'CHECKPOINT_PATH': '',
 'CHECKPOINT_SHA256': '3d7e2ba508986c7347380767233b017d4ac4c9fabc135f3457de8625dc833dca',
 'DOWNLOAD_CHECKPOINT': True,
 'HF_REVISION': 'de15dc5686a8628d051dae0a85b80d34ed690c68',
 'HF_FILENAME': 'ckpts/ckpt_msrvtt_retrieval_looseType/pytorch_model.bin.4',
 'EXPECTED_INPUT_SHA256': {'train_csv': '0a7ece2418e1f7ac5680555a04ddfef8950293ac3c4bdd6584fb0fa6cc0ff7d3',
                           'test_csv': 'e61144b312c8a2fc2126f96504c981f8c23c05b3ba9655a56e2af50ac8a4b16a',
                           'captions_json': '3540d91f9fd5f9a56d75b00c6576f30814fd19395d027791a6d438e9dc297014'},
 'EXPECTED_TRAIN_VIDEOS': 9000,
 'EXPECTED_TEST_VIDEOS': 1000,
 'EXPECTED_CAPTIONS_PER_VIDEO': 20,
 'EXPECTED_TRAIN_ORDER_SHA256': '722482c1fb6e560e594134ecf076a997701fbc0d10f5f0a7800ba17c0f5b1e27',
 'EXPECTED_RESERVED_ORDER_SHA256': 'd7ace5316b94212bd0fc58cc913d4137f8b324368632ea518e34e62d5343a900',
 'MEDIA_MANIFEST_PATH': '',
 'MEDIA_MANIFEST_FILENAME': 'msrvtt_train8100_v2.manifest.json',
 'EXPECTED_MEDIA_MANIFEST_SHA256': '3198e4b39f5c2c526430b48b620470144ce325198e174fbe357f5d898d4aa502',
 'REQUIRE_MEDIA_MANIFEST': True,
 'RUN_TRAIN': True,
 'SELECTION_POLICY': 'fixed_epochs',
 'EPOCHS': 5,
 'DEV_VAL_SIZE': 900,
 'SEED': 42,
 'PRECISION': 'fp32',
 'MAX_FRAMES': 12,
 'MAX_WORDS': 32,
 'ENCODER_BATCH': 4,
 'TEXT_BATCH': 1,
 'TRAIN_BATCH': 32,
 'LR': 0.001,
 'WEIGHT_DECAY': 0.0001,
 'TRAIN_LOGIT_SCALE': 20.0,
 'TAU': 0.05,
 'ALPHA': 0.25,
 'RESUME': True,
 'PREVIOUS_RUN_DIR': '',
 'ALLOW_CPU_DEBUG': False,
 'MIN_FREE_DISK_GIB': 2.0}
CONFIG["WORK_DIR"] = str(base / "public_training_run")


In [ ]:
# Thay payload nhúng bằng mã công khai có SHA256; input dữ liệu vẫn được gắn riêng.
import importlib.util

PUBLIC_CODE = Path("/kaggle/working/temporal-hoi-video-text-matching/scripts/kaggle/msrvtt")
_spec = importlib.util.spec_from_file_location("prepare_public_sources", PUBLIC_CODE / "prepare_sources.py")
_prepare = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_prepare)
_prepare.restore_sources(PUBLIC_CODE, CODE_DIR)
for _module in ("data_pipeline", "retrieval_core", "workflow", "resources", "bpe_bootstrap", "b3_probe_runtime", "fulltrain_runtime", "b3_eval_runtime"):
    sys.modules.pop(_module, None)
VENDOR_DIR = CODE_DIR / "vendor"
importlib.import_module("bpe_bootstrap").ensure_bpe(VENDOR_DIR / "bpe_simple_vocab_16e6.txt.gz", CONFIG["INPUT_ROOT"])
_prepare.capture_stage(PUBLIC_CODE, CODE_DIR, Path(CONFIG["WORK_DIR"]), "training", Path(CONFIG["INPUT_ROOT"]))
_work = Path(CONFIG["WORK_DIR"])
CONFIG["MEDIA_MANIFEST_PATH"] = str(CODE_DIR / CONFIG["MEDIA_MANIFEST_FILENAME"])
atomic_json = importlib.import_module("data_pipeline").atomic_json


In [ ]:
# Cell 4 — Kiểm đủ 8100 video/hash trước encoder hoặc training.
from fulltrain_runtime import prepare_fulltrain

FULLTRAIN_STATE = prepare_fulltrain(CONFIG)
print("Inventory:", json.dumps(FULLTRAIN_STATE["inventory"], ensure_ascii=False, indent=2))
atomic_json(_work / "metadata_inventory.json", FULLTRAIN_STATE["inventory"])
print("Preflight passed; no model training has started in this cell.")


In [ ]:
# Cell 5 — Cache đầy đủ và học head, encoder giữ đóng băng.
from fulltrain_runtime import run_cache_train

FULLTRAIN_REPORT = run_cache_train(CONFIG, VENDOR_DIR)
print("Status:", FULLTRAIN_REPORT["status"])
print(json.dumps({key: value for key, value in FULLTRAIN_REPORT.items()
                  if isinstance(value, (str, int, float, bool)) or value is None},
                 ensure_ascii=False, indent=2))
print("Đọc fulltrain_report.json để xem đầy đủ provenance và tài nguyên.")


In [ ]:
# Cell 6 — Xuất model/report; feature bundle được giữ riêng.
import zipfile

result_zip = base / "msrvtt_b3_train8100_results.zip"
if not (_work / "fulltrain_report.json").is_file():
    raise FileNotFoundError("Chưa có report; chạy Cell5 hoặc kiểm tra lỗi trước.")
_export_files = sorted(p for p in _work.glob("*.json") if p.is_file())
_export_files += sorted(p for p in (_work / "heads").glob("*")
                        if p.is_file() and p.suffix in {".pt", ".json"})
if (_work / "runtime_sources.zip").is_file():
    _export_files.append(_work / "runtime_sources.zip")
with zipfile.ZipFile(result_zip, "w", zipfile.ZIP_DEFLATED) as _archive:
    for _file in _export_files:
        _archive.write(_file, arcname=str(_file.relative_to(_work)))
print("Tải ZIP model + report:", result_zip)
_saved_report = json.loads((_work / "fulltrain_report.json").read_text("utf-8"))
_final_head = _work / "heads" / "final.pt"
if _saved_report.get("training_complete") and _final_head.is_file():
    print("Final head:", _final_head)
else:
    print("Run chưa hoàn tất; ZIP này dùng kiểm tra lỗi, chưa phải model cuối.")
if (_work / "train_features.pt").is_file():
    print("Giữ feature bundle để resume:", _work / "train_features.pt")
if importlib.util.find_spec("IPython") is not None:
    from IPython.display import FileLink, display
    display(FileLink(str(result_zip)))
